## Données: Villes et routes

In [ ]:
Villes = {
    "Montreal" : (45.30, -73.35),
    "Quebec" : (46.81, -71.21),
    "Ottawa" : (45.42, -75.70),
    "Toronto" : (43.65, -79.38),
    "Buffalo" : (42.89, -79.87),
    "Boston" : (42.36, -71.06),
    "New York" : (40.71, -74.01),
    "Chicago" : (41.88, -87.63)
}

Routes = [
    #Depuis Montreal
    ("Montreal", "Quebec", 250),
    ("Montreal", "Ottawa", 200),
    ("Montreal", "Boston", 435),
    ("Montreal", "New York", 595),
    #Depuis Quebec
    ("Quebec", "Montreal", 255),
    ("Quebec", "Boston", 650),
    #Depuis Ottawa
    ("Ottawa", "Montreal", 195),
    ("Ottawa", "Toronto", 450),
    #Depuis Toronto
    ("Toronto", "Ottawa", 445),
    ("Toronto", "Buffalo", 155),
    ("Toronto", "Chicago", 840),
    #Depuis Buffalo
    ("Buffalo", "Toronto", 160),
    ("Buffalo", "New York", 590),
    ("Buffalo", "Boston", 700),
    ("Buffalo", "Chicago", 860),
    #Depuis Boston
    ("Boston", "New York", 345),
    ("Boston", "Montreal", 440),
    ("Boston", "Buffalo", 695),
    #Depuis New York
    ("New York", "Boston", 350),
    ("New York", "Buffalo", 585),
    ("New York", "Chicago", 1270),
    #Depuis Chicago
    ("Chicago", "Toronto", 835),
    ("Chicago", "Buffalo", 855),
    ("Chicago", "New York", 1275)
]

## Graphe orienté pondéré

In [ ]:
class GraphOriente:
    def __init__(self, villes: dict):
        self.villes = villes
        self.adjacences = {ville: [] for ville in villes}
    
    def ajouter_arc(self, origine: str, destination: str, poids: int):
        self.adjacences[origine].append((destination, poids))
    
    def voisins(self, ville: str) -> list:
        return self.adjacences[ville]
    
    def sommets(self) -> list:
        return list(self.villes.keys())
    
    def coordonnees(self, ville: str) -> tuple:
        return self.villes[ville]
    
    def __repr__(self):
        lignes = []
        for ville, arcs in self.adjacences.items():
            for voisin, poids in arcs:
                lignes.append(f"  {ville} -> {voisin} : {poids} km")
        return "Graphe orienté:\n" + "\n".join(lignes)
        

graphe = GraphOriente(Villes)

for origine, destination, poids in Routes:
    graphe.ajouter_arc(origine, destination, poids)

print(graphe)

## Distance Euclidienne

In [ ]:
import math

def heuristique(graphe: GraphOriente, ville_a: str, ville_b: str) -> float:

    lat_a, lon_a = graphe.coordonnees(ville_a)
    lat_b, lon_b = graphe.coordonnees(ville_b)
    return math.sqrt((lat_a - lat_b) ** 2 + (lon_a - lon_b) ** 2)

h = heuristique(graphe, "Quebec", "Buffalo")
print(f"Heuristique Quebec -> Buffalo : {h:.4f}")

## Algorithme A*

In [ ]:
import heapq
import time

def a_star(graphe: GraphOriente, source: str, destination: str):

    debut = time.perf_counter()

    g = {ville: float('inf') for ville in graphe.sommets()}
    g[source] = 0

    predecesseurs = {ville: None for ville in graphe.sommets()}

    h_source = heuristique(graphe, source, destination)
    file = [(h_source, source)]

    visites = set()
    noeuds_explores = 0

    while file:
        f_actuel, ville_actuelle = heapq.heappop(file)

        if ville_actuelle in visites:
            continue

        visites.add(ville_actuelle)
        noeuds_explores += 1

        if ville_actuelle == destination:
            break

        for voisin, poids in graphe.voisins(ville_actuelle):
            if voisin in visites:
                continue

            g_nouveau = g[ville_actuelle] + poids

            if g_nouveau < g[voisin]:
                g[voisin] = g_nouveau
                predecesseurs[voisin] = ville_actuelle
                f_voisin = g_nouveau + heuristique(graphe, voisin, destination)
                heapq.heappush(file, (f_voisin, voisin))

    fin = time.perf_counter()
    temps_exec = fin - debut

    chemin = []
    etape = destination
    while etape is not None:
        chemin.append(etape)
        etape = predecesseurs[etape]
    chemin.reverse()

    if chemin[0] != source:
        return None, float('inf'), noeuds_explores, temps_exec

    cout_total = g[destination]

    return chemin, cout_total, noeuds_explores, temps_exec

chemin, cout, noeuds, temps = a_star(graphe, "Montreal", "Chicago")

print("=== Résultats A* ===")
print(f"Chemin        : {' -> '.join(chemin)}")
print(f"Coût total    : {cout} km")
print(f"Nœuds explorés: {noeuds}")
print(f"Temps d'exec  : {temps:.6f} secondes")